In [1]:
import pandas as pd
import numpy as np
import pickle
import logging
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
def create_loan_dataset(n_samples=100000):
    np.random.seed(42)
    data = {
        'income': np.random.randint(10000, 200000, size=n_samples),
        'employment_status': np.random.choice(['Employed', 'Self-Employed', 'Unemployed'], size=n_samples, p=[0.6, 0.3, 0.1]),
        'credit_score': np.random.randint(300, 901, size=n_samples),
        'existing_loan_count': np.random.randint(0, 6, size=n_samples),
        'employment_length': np.random.randint(0, 31, size=n_samples),
        'loan_amount_requested': np.random.randint(5000, 500000, size=n_samples),
        'savings': np.random.randint(0, 100000, size=n_samples),
        'dependents': np.random.randint(0, 6, size=n_samples),
    }
    df = pd.DataFrame(data)

    approval_score = (
        (df['credit_score'] - 300) / 600 * 0.4 +
        (df['income'] / 200000) * 0.25 -
        (df['loan_amount_requested'] / 500000) * 0.2 +
        (df['savings'] / 100000) * 0.1 -
        (df['existing_loan_count'] / 5) * 0.15 +
        (df['employment_length'] / 30) * 0.1 +
        (df['employment_status'] == 'Employed').astype(int) * 0.1 -
        (df['employment_status'] == 'Unemployed').astype(int) * 0.2 -
        (df['dependents'] / 5) * 0.1
    )
    noise = np.random.normal(0, 0.08, size=n_samples)
    df['approved'] = ((approval_score + noise) > 0.35).astype(int)

    return df

In [3]:
logging.info("Generating synthetic loan dataset...")
df = create_loan_dataset()
df.head()

2026-07-11 08:19:34,784 - INFO - Generating synthetic loan dataset...


,income,employment_status,credit_score,existing_loan_count,employment_length,loan_amount_requested,savings,dependents,approved
0,131958,Employed,304,3,5,163880,88609,4,0
1,156867,Employed,487,2,29,435699,67826,1,0
2,141932,Self-Employed,896,4,20,197044,80115,2,1
3,113694,Employed,720,4,19,239412,46084,5,0
4,129879,Employed,770,4,23,333618,50092,4,1


In [4]:
le_employment = LabelEncoder()
df['employment_status_enc'] = le_employment.fit_transform(df['employment_status'])
df[['employment_status', 'employment_status_enc']].drop_duplicates()

,employment_status,employment_status_enc
0,Employed,0
2,Self-Employed,1
19,Unemployed,2


In [5]:
features = ['income', 'employment_status_enc', 'credit_score', 'existing_loan_count',
            'employment_length', 'loan_amount_requested', 'savings', 'dependents']
X = df[features]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 80000, Test size: 20000


In [6]:
model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
logging.info("Training model...")
model.fit(X_train, y_train)
logging.info("Training complete.")

2026-07-11 08:19:34,970 - INFO - Training model...
2026-07-11 08:19:57,312 - INFO - Training complete.


In [7]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.94      0.91     13704
           1       0.84      0.74      0.79      6296

    accuracy                           0.88     20000
   macro avg       0.86      0.84      0.85     20000
weighted avg       0.87      0.88      0.87     20000



In [8]:
with open('loan_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('label_encoder_property.pkl', 'wb') as f:
    pickle.dump(le_employment, f)

logging.info("Model and encoder saved.")

2026-07-11 08:19:58,068 - INFO - Model and encoder saved.
